In part two of this project, you’ll refactor the code to follow a more structured object-oriented approach. We’ll create a Student model as a base class for managing individual students, and a StudentManagementSystem class that encapsulates all the CRUD operations, as well as the logic for handling multiple students. The data will be saved to and loaded from a JSON file using methods in the StudentManagementSystem class. Let’s break it down step-by-step:

Project Structure

Create a directory for the project:


In [1]:
!student_management_system/
!├── student.py              # Defines the Student class
!├── management_system.py     # Defines the StudentManagementSystem class
!└── main.py                  # Main script to run the system

/bin/bash: line 1: student_management_system/: No such file or directory
/bin/bash: line 1: ├──: command not found
/bin/bash: line 1: ├──: command not found
/bin/bash: line 1: └──: command not found



1.	student.py: Contains the Student model, representing individual student data and methods.
2.	management_system.py: Contains the StudentManagementSystem class for managing CRUD operations, average calculations, and data persistence.
3.	main.py: Script to interact with the student management system, running the application.

student.py - Define the Student Model

In this file, we define a Student class that handles individual student data, such as subjects and grades.



In [2]:
# student_management_system/student.py

from typing import Dict

class Student:
    def __init__(self, name: str, grades: Dict[str, int]):
        """
        Initialize a student with a name and their grades.

        Parameters:
            name (str): Name of the student.
            grades (dict): A dictionary of subjects and grades.
        """
        self.name = name
        self.grades = grades

    def add_grade(self, subject: str, grade: int) -> None:
        """Add or update a grade for a specific subject."""
        self.grades[subject] = grade

    def get_average_grade(self) -> float:
        """Calculate the average grade for the student."""
        average_grade = sum(self.grades.values()) / len(self.grades.values())

        return average_grade

    def to_dict(self) -> Dict[str, any]:
        """Convert the Student instance to a dictionary for JSON serialization."""
        return {self.name: {"grades": self.grades}}

    @staticmethod
    def from_dict(data: Dict[str, any]) -> "Student":
        """Create a Student instance from a dictionary."""
        for student, grades in data.items():
            return Student(student, grades["grades"])

In [3]:
# =================================
# Test result, with Ivy from part 1
# =================================

# Testing the constructor
Ivy = Student(name="Ivy", grades={"Math": 85, "Science": 90})
print(f"Hello, {Ivy.name}")
print(f"Your grade is {Ivy.grades}" , "\n")

# Testing get_average_grade method
print(f"The average score: {Ivy.get_average_grade()}", "\n")

# Testing to_dict method
data = Ivy.to_dict()
print("In a dictionary foramt: ", data, "\n")

# Testing from_dict method
print("Creating Ivy from data ... ")
Ivy = Student.from_dict(data)
print(f"Hello, {Ivy.name}")
print(f"Your grade is {Ivy.grades}")


Hello, Ivy
Your grade is {'Math': 85, 'Science': 90} 

The average score: 87.5 

In a dictionary foramt:  {'Ivy': {'grades': {'Math': 85, 'Science': 90}}} 

Creating Ivy from data ... 
Hello, Ivy
Your grade is {'Math': 85, 'Science': 90}



management_system.py - Define the Student Management System

This file contains the StudentManagementSystem class, which manages a collection of Student objects and includes methods for CRUD operations, saving/loading data, and calculating averages.



In [4]:
# student_management_system/management_system.py

import json
from typing import Dict, List
from student import Student

class StudentManagementSystem:
    def __init__(self):
        """Initialize the student management system with an empty dictionary of students."""
        self.students = {}

    def add_student(self, name: str, grades: Dict[str, int]) -> bool:
        """Add a new student to the system."""
            # if the student already exists, return False
        if name in self.students:
            return False

        # if grades are not in the correct format, return False
        if not isinstance(grades, dict):
            return False

        # if subject and grade are not in the correct format, return False
        for subject, grade in grades.items():
            if not isinstance(subject, str) or not isinstance(grade, int):
                return False

        # else, add students and return True
        else:
            self.students[name] = Student(name, grades)
            return True

    def update_grade(self, name: str, subject: str, grade: int) -> bool:
        """Update a specific student's grade for a given subject."""
            # Return false if student was not found
        if name not in self.students:
            return False

        # Return false if the subject was not found
        if subject not in self.students[name].grades:
            return False

        # else, update the grade and return True
        self.students[name].add_grade(subject, grade)
        return True

    def delete_student(self, name: str) -> bool:
        """Delete a student from the system."""
            # Reutrn false if the student was not found
        if name not in self.students:
            return False

        # else, delete student and return true
        del self.students[name]

        return True

    def display_records(self) -> None:
        """Display all student records with their grades and average grades."""
        if not self.students:
            print("No student records found.")
            return


        # Make a dictionary where keys are subjects and values are lists of grades for each student
        grades_per_subject = {}

        # Display a student's record and their average grades
        for student in self.students.values():
            print(f"Name: {student.name}")
            print(f"Grades: {student.grades}")
            print(f"Average: {student.get_average_grade():.2f}")
            print("-" * 30)

            for subject, grade in student.grades.items():
                if subject not in grades_per_subject:
                    grades_per_subject[subject] = []
                grades_per_subject[subject].append(grade)



        # Define helper function for calculating average grades
        get_avg = lambda x: round(sum(x) / len(x), 1)

        # Calculate average for each subject
        avg_per_subject = {subject: get_avg(grades) for subject, grades in grades_per_subject.items()}

        print(f"Average grade for each subject:\n{avg_per_subject}")




    def save_to_file(self, filename: str = "students.json") -> bool:
        """Save all students to a JSON file."""
        try:
            data = {}

            for student in self.students.values():
                data.update(student.to_dict())

            with open(filename, "w", encoding="utf-8") as file:
                json.dump(data, file, indent=4)

            return True

        except (OSError, TypeError):
            return False

    def load_from_file(self, filename: str = "students.json") -> bool:
        """Load student data from a JSON file."""
        try:
            with open(filename, "r", encoding="utf-8") as file:
                data = json.load(file)

            self.students = {}

            for student_data in data.values():
                # Student.from_dict() expects {name: {"grades": ...}}
                # so we need to pass each student entry back in that format.
                pass

            for name, student_data in data.items():
                self.students[name] = Student(
                    name,
                    student_data["grades"]
                )

            return True

        except (OSError, json.JSONDecodeError, KeyError, TypeError):
            return False


main.py - Main Script to Run the System

In this script, you’ll interact with the StudentManagementSystem to manage student data.



In [5]:
# student_management_system/main.py

from management_system import StudentManagementSystem

def main():
    system = StudentManagementSystem()

    # Load existing data if available
    system.load_from_file()

    # Example operations
    system.add_student("Alice", {"Math": 85, "Science": 90})
    system.add_student("Bob", {"English": 78, "History": 82})

    system.update_grade("Alice", "Math", 92)
    system.update_grade("Bob", "History", 88)

    system.display_records()

    system.delete_student("Bob")

    # Save data
    system.save_to_file()

if __name__ == "__main__":
    main()

Name: Alice
Grades: {'Math': 92, 'Science': 90}
Average: 91.00
------------------------------
Name: Bob
Grades: {'English': 78, 'History': 88}
Average: 83.00
------------------------------
Average grade for each subject:
{'Math': 92.0, 'Science': 90.0, 'English': 78.0, 'History': 88.0}



### Summary

In this refactored project, we introduced object-oriented principles:

1.	Student Class: Represents an individual student with methods to manage grades and calculate averages.
2.	StudentManagementSystem Class: Encapsulates CRUD operations, average calculations, and data persistence for a collection of students.
3.	main.py: The main script to run the program and interact with the StudentManagementSystem.

This structure makes the code modular, reusable, and easier to maintain.

### How to Submit the Project

1. Create a Jupyter Notebook file on your Google Drive.
2. Write your solution in the notebook.
3. Share access with your mentor (or provide a link with editing rights).
4. Submit the link to the notebook file or the folder as your solution for this lesson.